In [11]:
incident = {
    "pipeline_name": "customer_data_pipeline",
    "error": "KeyError: 'customer_id'",
    "expected_columns": [
        "customer_id",
        "name",
        "email",
        "signup_date"
    ],
    "actual_columns": [
        "id",
        "name",
        "email",
        "signup_date"
    ]
}

In [12]:
def diagnose_schema(incident):
    expected = set(incident["expected_columns"])
    actual = set(incident["actual_columns"])

    print(expected)
    print(actual)

    missing_columns = sorted(expected - actual)
    unexpected_columns = sorted(actual - expected)

    print(missing_columns)
    print(unexpected_columns)

    if missing_columns or unexpected_columns:
        status = "failed"
    else:
        status = "passed"

    return {
        "pipeline_name": incident["pipeline_name"],
        "status": status,
        "missing_columns": missing_columns,
        "unexpected_columns": unexpected_columns
    }

In [13]:
result = diagnose_schema(incident)
print(result)

{'signup_date', 'email', 'name', 'customer_id'}
{'id', 'email', 'name', 'signup_date'}
['customer_id']
['id']
{'pipeline_name': 'customer_data_pipeline', 'status': 'failed', 'missing_columns': ['customer_id'], 'unexpected_columns': ['id']}


In [15]:
print("Pipeline:", result["pipeline_name"])
print("Status:", result["status"])
print("Missing columns:", result["missing_columns"])
print("Unexpected columns:", result["unexpected_columns"])

Pipeline: customer_data_pipeline
Status: failed
Missing columns: ['customer_id']
Unexpected columns: ['id']


In [17]:
from langchain_core.prompts import ChatPromptTemplate

In [18]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a data pipeline troubleshooting assistant.
        Explain pipeline failures using the provided diagnostic findings.
        Separate verified facts from possible causes.
        Never present an unverified cause as a confirmed fact.
        Recommend practical checks for the engineer."""
    ),
    (
        "human",
        """Analyze this pipeline incident:

        Pipeline: {pipeline_name}
        Error: {error}
        Diagnostic findings: {diagnostic_findings}

        Explain what is known, what might have happened,
        and what the engineer should check next."""
    )
])

In [19]:
prompt_value = prompt.invoke({
    "pipeline_name": incident["pipeline_name"],
    "error": incident["error"],
    "diagnostic_findings": str(result)
})

print(prompt_value)

messages=[SystemMessage(content='You are a data pipeline troubleshooting assistant.\n        Explain pipeline failures using the provided diagnostic findings.\n        Separate verified facts from possible causes.\n        Never present an unverified cause as a confirmed fact.\n        Recommend practical checks for the engineer.', additional_kwargs={}, response_metadata={}), HumanMessage(content="Analyze this pipeline incident:\n\n        Pipeline: customer_data_pipeline\n        Error: KeyError: 'customer_id'\n        Diagnostic findings: {'pipeline_name': 'customer_data_pipeline', 'status': 'failed', 'missing_columns': ['customer_id'], 'unexpected_columns': ['id']}\n\n        Explain what is known, what might have happened,\n        and what the engineer should check next.", additional_kwargs={}, response_metadata={})]


In [20]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
from langchain_openai import ChatOpenAI

In [22]:
llm = ChatOpenAI(
    model = "openai/gpt-oss-20b",
    api_key = os.getenv("GROK_API_KEY"),
    base_url = "https://api.groq.com/openai/v1",
)

In [23]:
chain = prompt | llm

In [24]:
response = chain.invoke({
    "pipeline_name": incident["pipeline_name"],
    "error": incident["error"],
    "diagnostic_findings": str(result)
})

print(response.content)

**Verified Facts**

| Item | Detail |
|------|--------|
| **Pipeline** | `customer_data_pipeline` |
| **Status** | Failed |
| **Error** | `KeyError: 'customer_id'` (raised during execution) |
| **Diagnostic findings** | • `missing_columns`: `['customer_id']`<br>• `unexpected_columns`: `['id']` |

The pipeline attempted to read a column named `customer_id`, but that column was not present in the source data. Instead, a column named `id` was found.

---

**Possible (Unverified) Causes**

1. **Source schema change** – The upstream source (e.g., a database table, CSV, or JSON file) may have renamed the `customer_id` column to `id`.  
2. **Data file mix‑up** – The pipeline could be pulling the wrong file or dataset that uses a different column name.  
3. **Configuration mismatch** – The pipeline’s schema definition or mapping configuration still expects `customer_id`, while the actual data uses `id`.  
4. **ETL transformation error** – A prior transformation step may have inadvertently drop

In [26]:
from pydantic import BaseModel, Field


class PipelineDiagnosis(BaseModel):
    incident_summary: str = Field(
        description="A short summary of the pipeline failure"
    )
    observed_facts: list[str] = Field(
        description="Facts supported by the incident data and diagnostics"
    )
    likely_causes: list[str] = Field(
        description="Possible causes that have not necessarily been confirmed"
    )
    supporting_evidence: list[str] = Field(
        description="Evidence from the incident that supports the analysis"
    )
    recommended_checks: list[str] = Field(
        description="Practical checks an engineer should perform next"
    )
    unknowns: list[str] = Field(
        description="Important information that is still unavailable"
    )

In [27]:
structured_llm = llm.with_structured_output(PipelineDiagnosis)

In [28]:
diagnosis_chain = prompt | structured_llm

In [29]:
diagnosis = diagnosis_chain.invoke({
    "pipeline_name": incident["pipeline_name"],
    "error": incident["error"],
    "diagnostic_findings": str(result)
})

print(diagnosis)

incident_summary="Pipeline failed due to a missing 'customer_id' column." observed_facts=['Pipeline name: customer_data_pipeline', 'Status: failed', "Missing columns reported: ['customer_id']", "Unexpected columns reported: ['id']"] likely_causes=["Schema change in source data dropping 'customer_id'", "Column renamed from 'customer_id' to 'id' without updating the pipeline", "Data source now provides an 'id' column instead of 'customer_id'", "Pipeline mapping or configuration still references 'customer_id'"] supporting_evidence=["Error: KeyError: 'customer_id'", "Diagnostic findings contain missing_columns ['customer_id'] and unexpected_columns ['id']"] recommended_checks=["Verify the upstream source schema for the presence of 'customer_id' column", "Check recent changes to the source data or transformations that might rename 'customer_id' to 'id'", 'Inspect pipeline configuration or mapping files to ensure they reference the correct column name', 'Run a schema validation step against 

In [32]:
print("Summary:", diagnosis.incident_summary)

print("\nObserved Facts:")
for fact in diagnosis.observed_facts:
    print("-", fact)

print("\nLikely Causes:")
for cause in diagnosis.likely_causes:
    print("-", cause)

print("\nRecommended Checks:")
for check in diagnosis.recommended_checks:
    print("-", check)

print("\nUnknowns:")
for unknown in diagnosis.unknowns:
    print("-", unknown)

Summary: Pipeline failed due to a missing 'customer_id' column.

Observed Facts:
- Pipeline name: customer_data_pipeline
- Status: failed
- Missing columns reported: ['customer_id']
- Unexpected columns reported: ['id']

Likely Causes:
- Schema change in source data dropping 'customer_id'
- Column renamed from 'customer_id' to 'id' without updating the pipeline
- Data source now provides an 'id' column instead of 'customer_id'
- Pipeline mapping or configuration still references 'customer_id'

Recommended Checks:
- Verify the upstream source schema for the presence of 'customer_id' column
- Check recent changes to the source data or transformations that might rename 'customer_id' to 'id'
- Inspect pipeline configuration or mapping files to ensure they reference the correct column name
- Run a schema validation step against the incoming data to capture missing/extra columns before processing
- Confirm that downstream components expect 'id' or 'customer_id' and adjust accordingly

Unknow